# LimX HU_D03 locomotion training on Kaggle (mjlab)

Trains `Mjlab-Velocity-Flat-LimX-HU-D03`: the HU_D03 humanoid learns to follow forward, sideways and turning
velocity commands on flat ground, with [mjlab](https://github.com/mujocolab/mjlab) and the
[LimX HU_D03 model](https://github.com/limxdynamics/humanoid-description).

### Before running

1. **Settings → Accelerator → GPU T4 x2.** Not P100: it is older than Turing and mjlab does not support it.
2. **Settings → Internet → On** (needs a phone-verified Kaggle account).

### How to run

* **Try it interactively first:** run the cells down to *4. Sanity checks* and make sure the stand video looks right.
  Interactive sessions stop when the browser is idle, so do not train there.
* **Train with Save Version → Save & Run All (Commit).** That runs the whole notebook in the background
  (up to 12 h) and keeps everything in `/kaggle/working`: checkpoints, ONNX policy, training curves, video and a zip.
  Training stops itself after `TIME_BUDGET_HOURS` so the steps after it still run.

### Continue training in a new version

Open the notebook, *Add Input → Your Work → Notebooks*, and add **this notebook's previous version**.
The settings cell finds its newest checkpoint under `/kaggle/input` and resumes from it.

### About the model

The real HU_D03 drives its ankles and waist through parallel linkages (two motors per joint pair). Here they are
serial `ankle_pitch/roll` and `waist_roll/pitch` joints, as mjlab does for the Unitree G1, so the policy's targets
must be converted to A/B motor commands on the real robot. LimX does not publish PD gains; they are derived from
the rotor inertias (10 Hz, damping ratio 2).

## 1. Settings

In [ ]:
import os, shutil, signal, subprocess
from pathlib import Path

WORK = Path("/kaggle/working/hu_d03_rl")   # Saved with each version.
CODE = Path("/kaggle/temp/hu_d03_code")    # Installed code, re-created each run.
WORK.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

NUM_ENVS = 4096            # Use 2048 if the GPU runs out of memory.
MAX_ITERATIONS = 10_000    # Upper bound; the time budget usually stops training first.
TIME_BUDGET_HOURS = 10.5   # Kaggle stops a version at 12 h; leave time for the video and zip.
USE_TWO_GPUS = False       # True splits envs over both T4s (faster, but less tested).

# Code versions. These commits were tested with this notebook; "main" gets the latest.
MJLAB_REF = "8bb2bc9e187797689ec82c78c35019afd2fd0376"
DESCRIPTION_REF = "0dd8454384c58991f3e70fd7613be29006d39f86"

TASK = "Mjlab-Velocity-Flat-LimX-HU-D03"
EXPERIMENT = "hu_d03_velocity"
LOG_ROOT = WORK / "logs" / "rsl_rl" / EXPERIMENT

os.environ["WANDB_MODE"] = "disabled"
os.environ["MUJOCO_GL"] = "egl"
os.environ["MJLAB_WARP_QUIET"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"  # Stream training logs into the notebook.
os.environ["PYTHONFAULTHANDLER"] = "1"  # Print a Python traceback on a crash.
os.environ["HU_D03_DESCRIPTION_DIR"] = str(CODE / "humanoid-description" / "HU_D03_description")

# Resume: newest checkpoint of an earlier version added under "Input".
def newest_input_run():
  ckpts = list(Path("/kaggle/input").glob(f"**/logs/rsl_rl/{EXPERIMENT}/*/model_*.pt"))
  if not ckpts:
    return None
  ckpts.sort(key=lambda p: (p.parent.name, int(p.stem.split("_")[1])))
  return ckpts[-1].parent

RESUME_RUN = newest_input_run()
if RESUME_RUN is not None:
  dest = LOG_ROOT / RESUME_RUN.name
  if not dest.exists():
    shutil.copytree(RESUME_RUN, dest)
  print("will resume from", dest, "| newest checkpoint:",
        max(dest.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1])).name)
else:
  print("no earlier run under /kaggle/input: training from scratch")

## 2. Check the GPU

In [ ]:
out = subprocess.run(
  ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total", "--format=csv,noheader"],
  capture_output=True, text=True,
)
if out.returncode != 0:
  raise SystemExit("No GPU. Set Settings → Accelerator → GPU T4 x2.")
print(out.stdout)
name, cap, driver, _ = [s.strip() for s in out.stdout.splitlines()[0].split(",")]
if float(cap) < 7.5:
  raise SystemExit(f"{name} (compute {cap}) is too old for mjlab. Pick GPU T4 x2, not P100.")
CUDA_EXTRA = "cu130" if int(driver.split(".")[0]) >= 580 else "cu126"
NUM_GPUS = len(out.stdout.splitlines())
print("mjlab extra:", CUDA_EXTRA, "| GPUs:", NUM_GPUS)

## 3. Install mjlab and the HU_D03 model (about 3–5 minutes)

In [ ]:
import threading, time

def sh(cmd, cwd=None, max_seconds=None, check=True, log_file=None, on_line=None, on_tick=None):
  """Run a shell command and stream its output into the notebook.

  log_file: write the output there instead of the notebook (the last lines are
  shown if the command fails). on_line(line) sees every output line;
  on_tick(elapsed_s) is called every few seconds from the notebook thread.
  After max_seconds the command gets Ctrl-C (then a kill) and counts as done.
  """
  print("$", cmd, flush=True)
  log = open(log_file, "a") if log_file else None
  proc = subprocess.Popen(cmd, shell=True, cwd=cwd, text=True, bufsize=1,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          start_new_session=True)
  sh.last_output = []

  def pump_output():
    for line in proc.stdout:
      if log:
        log.write(line)
      else:
        print(line, end="", flush=True)
      if on_line:
        on_line(line)
      sh.last_output.append(line)
      del sh.last_output[:-200]

  pump = threading.Thread(target=pump_output)
  pump.start()
  start, timed_out = time.time(), False
  while proc.poll() is None:
    if on_tick:
      on_tick(time.time() - start)
    if max_seconds is not None and time.time() - start > max_seconds:
      timed_out = True
      print(f"\n[time budget reached after {max_seconds / 3600:.1f} h: stopping]", flush=True)
      os.killpg(proc.pid, signal.SIGINT)
      try:
        proc.wait(timeout=120)
      except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
      break
    time.sleep(10 if on_tick else 0.5)
  pump.join(timeout=30)
  if on_tick:
    on_tick(time.time() - start)
  if log:
    log.close()
  if check and proc.returncode != 0 and not timed_out:
    if log:
      print("".join(sh.last_output[-60:]))
    raise SystemExit(f"command failed with exit code {proc.returncode}")
  return proc.returncode

def fetch(repo, ref, dest):
  if not (dest / ".git").exists():
    sh(f"git init -q {dest} && git -C {dest} remote add origin {repo}")
  sh(f"git -C {dest} fetch -q --depth 1 origin {ref} && git -C {dest} checkout -q FETCH_HEAD")
  sh(f"git -C {dest} log -1 --format='%h %cd %s' --date=short")

fetch("https://github.com/mujocolab/mjlab.git", MJLAB_REF, CODE / "mjlab")
fetch("https://github.com/limxdynamics/humanoid-description.git", DESCRIPTION_REF, CODE / "humanoid-description")

sh("pip install -q uv")
sh(f"uv pip install --system -q -e '{CODE / 'mjlab'}[{CUDA_EXTRA}]'")

### Write the HU_D03 package

Generated from the `hu-d03-rl` folder (`notebooks/build_kaggle_locomotion.py`). You can also edit the cells here.

In [ ]:
for d in ['scripts', 'src/hu_d03_mjlab', 'src/hu_d03_mjlab/robot', 'src/hu_d03_mjlab/tasks']:
  (Path('/kaggle/temp/hu_d03_code/hu-d03-rl') / d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/pyproject.toml
[project]
name = "hu-d03-mjlab"
version = "0.1.0"
description = "LimX HU_D03 locomotion and motion-tracking tasks for mjlab"
requires-python = ">=3.10"
dependencies = ["mjlab"]

[project.entry-points."mjlab.tasks"]
hu_d03 = "hu_d03_mjlab.tasks"

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/__init__.py
"""LimX HU_D03 tasks for mjlab."""

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/__init__.py
"""LimX HU_D03 robot model for mjlab."""

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_spec.py
"""Load the LimX HU_D03 MJCF and turn it into a serial-joint model for mjlab.

This module only depends on `mujoco`, so it can be tested on a CPU-only
machine without mjlab installed.

What the conversion does:
  * Removes the parallel ankle (A/B achilles) and waist (A/B) linkages and the 6
    `<connect>` constraints that close them. The policy then commands the
    serial joints `ankle_pitch/roll` and `waist_roll/pitch` directly, like the
    Unitree G1 model in mjlab. The linkage mass is added to the parent body.
  * Removes the floor, light, camera, actuators and sensors from the source XML
    (mjlab adds its own terrain, actuators and sensors).
  * Sets every geom margin to 0 (MuJoCo Warp does not support margins on
    box/mesh pairs, such as the two foot boxes).
  * Names every collision geom (`<body>_collision<i>`, feet are
    `left_foot_collision` / `right_foot_collision`) so mjlab can match them.
  * Adds foot sites at the sole and the IMU sensors mjlab's tasks read.
"""

import os
from pathlib import Path

import mujoco
import numpy as np

_DEFAULT_DESCRIPTION_DIR = (
  Path(__file__).resolve().parents[3]
  / "third_party"
  / "humanoid-description"
  / "HU_D03_description"
)


def description_dir() -> Path:
  """Directory of `HU_D03_description` (override with $HU_D03_DESCRIPTION_DIR)."""
  return Path(os.environ.get("HU_D03_DESCRIPTION_DIR", _DEFAULT_DESCRIPTION_DIR))


def source_xml() -> Path:
  xml = description_dir() / "xml" / "HU_D03_03.xml"
  if not xml.exists():
    raise FileNotFoundError(
      f"{xml} not found. Clone https://github.com/limxdynamics/humanoid-description "
      "and set HU_D03_DESCRIPTION_DIR to its HU_D03_description folder."
    )
  return xml


# Bodies (with their subtrees) that form the closed-loop linkages.
LINKAGE_BODIES = (
  "left_A_achilles_link",
  "left_B_achilles_link",
  "right_A_achilles_link",
  "right_B_achilles_link",
  "waist_A_link",
  "waist_B_link",
)

# Foot sole: the foot box is centred 0.0535 m below the ankle roll frame and is
# 0.01 m half-thick, so the sole is at z = -0.0635.
FOOT_SITE_POS = (0.018, 0.0, -0.0635)
FOOT_GEOMS = {"left_foot": "left_foot_collision", "right_foot": "right_foot_collision"}


def _subtree_mass(body: mujoco.MjsBody) -> float:
  mass = body.mass
  for child in body.bodies:
    mass += _subtree_mass(child)
  return mass


def _add_mass(body: mujoco.MjsBody, extra: float) -> None:
  """Add point-like mass to a body, scaling its inertia by the same ratio."""
  ratio = (body.mass + extra) / body.mass
  body.mass = body.mass + extra
  body.inertia = np.asarray(body.inertia) * ratio


def get_spec() -> mujoco.MjSpec:
  spec = mujoco.MjSpec.from_file(str(source_xml()))

  # Equality constraints, actuators and sensors all reference linkage joints.
  for eq in list(spec.equalities):
    spec.delete(eq)
  for act in list(spec.actuators):
    spec.delete(act)
  for sensor in list(spec.sensors):
    spec.delete(sensor)

  # Floor, light and tracking camera live in the worldbody; mjlab adds terrain.
  for geom in list(spec.worldbody.geoms):
    spec.delete(geom)
  for light in list(spec.worldbody.lights):
    spec.delete(light)
  for cam in list(spec.worldbody.cameras):
    spec.delete(cam)

  # Remove the linkages, moving their mass to the parent body.
  for name in LINKAGE_BODIES:
    body = spec.body(name)
    parent = body.parent
    _add_mass(parent, _subtree_mass(body))
    spec.delete(body)

  # The free joint has a tiny spring/damper in the source XML; drop them.
  root = spec.body("base_link")
  free = root.joints[0]
  free.damping = np.zeros_like(free.damping)
  free.stiffness = np.zeros_like(free.stiffness)

  # Name collision geoms so mjlab's regex-based configs can find them.
  for body in spec.bodies:
    i = 0
    for geom in body.geoms:
      # The XML default gives every geom margin=0.001. MuJoCo Warp rejects a
      # non-zero margin on box/mesh pairs (e.g. foot-foot) with MULTICCD.
      geom.margin = 0.0
      if geom.contype == 0 and geom.conaffinity == 0:
        continue  # Visual mesh.
      if geom.name in FOOT_GEOMS:
        geom.name = FOOT_GEOMS[geom.name]
      elif not geom.name:
        geom.name = f"{body.name}_collision{i}"
        i += 1

  for side in ("left", "right"):
    spec.body(f"{side}_ankle_roll_link").add_site(
      name=f"{side}_foot", pos=FOOT_SITE_POS, size=(0.01, 0.01, 0.01), group=5
    )

  # Sensors read by mjlab's velocity and tracking tasks.
  spec.add_sensor(name="imu_ang_vel", type=mujoco.mjtSensor.mjSENS_GYRO,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_vel", type=mujoco.mjtSensor.mjSENS_VELOCIMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_acc", type=mujoco.mjtSensor.mjSENS_ACCELEROMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_upvector", type=mujoco.mjtSensor.mjSENS_FRAMEZAXIS,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="world",
                  reftype=mujoco.mjtObj.mjOBJ_SITE, refname="imu")
  spec.add_sensor(name="root_angmom", type=mujoco.mjtSensor.mjSENS_SUBTREEANGMOM,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="base_link")

  return spec

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_params.py
"""HU_D03 actuator and pose parameters (plain Python, no mjlab import).

Armatures are the reflected rotor inertias from the LimX SRDF
(`rotor mass * gear_ratio**2`), which match the `armature` values in the MJCF.
Effort limits are the `ctrlrange` of the MJCF motors.

LimX does not publish the PD gains used on the real robot, so stiffness and
damping follow mjlab's G1 rule: a second-order system with natural frequency
NATURAL_FREQ_HZ and damping ratio DAMPING_RATIO for the joint's armature.
Replace them with the real gains if you get them from LimX.

The ankle and waist roll/pitch joints are driven by two motors through a
parallel linkage. As for the G1, we assume a nominal 1:1 linkage, so these
joints get the armature and effort of both motors.
"""

import math
from dataclasses import dataclass

NATURAL_FREQ_HZ = 10.0
DAMPING_RATIO = 2.0

ARMATURE_LEG = 0.000244114 * 25.0**2  # 0.1526, hips and knees.
ARMATURE_36 = 0.000073217 * 36.0**2  # 0.0949, ankle/waist motors.
ARMATURE_ARM = 0.000073217 * 25.0**2  # 0.0458, shoulders and elbows.
ARMATURE_SMALL = 0.000017 * 25.0**2  # 0.0106, wrists, hands and head.


@dataclass(frozen=True)
class ActuatorGroup:
  joint_names_expr: tuple[str, ...]
  armature: float
  effort_limit: float

  @property
  def stiffness(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return self.armature * w**2

  @property
  def damping(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return 2.0 * DAMPING_RATIO * self.armature * w


ACTUATOR_GROUPS = {
  "leg": ActuatorGroup(
    (".*_hip_pitch_joint", ".*_hip_roll_joint", ".*_hip_yaw_joint", ".*_knee_joint"),
    ARMATURE_LEG,
    120.0,
  ),
  "ankle": ActuatorGroup(
    (".*_ankle_pitch_joint", ".*_ankle_roll_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "waist_yaw": ActuatorGroup(("waist_yaw_joint",), ARMATURE_36, 45.0),
  "waist": ActuatorGroup(
    ("waist_roll_joint", "waist_pitch_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "arm": ActuatorGroup(
    (
      ".*_shoulder_pitch_joint",
      ".*_shoulder_roll_joint",
      ".*_shoulder_yaw_joint",
      ".*_elbow_joint",
    ),
    ARMATURE_ARM,
    30.0,
  ),
  "small": ActuatorGroup(
    (".*_wrist_yaw_joint", ".*_wrist_pitch_joint", ".*_hand_yaw_joint", "head_.*"),
    ARMATURE_SMALL,
    18.0,
  ),
}

# Standing pose with slightly bent knees. Signs: negative hip pitch swings the
# thigh forward, positive knee flexes, negative elbow flexes. The base height
# puts the soles on the ground (checked by scripts/stand_test.py).
HOME_BASE_HEIGHT = 0.889
HOME_JOINT_POS = {
  ".*_hip_pitch_joint": -0.25,
  ".*_knee_joint": 0.5,
  ".*_ankle_pitch_joint": -0.25,
  "left_shoulder_roll_joint": 0.15,
  "right_shoulder_roll_joint": -0.15,
  ".*_elbow_joint": -0.3,
}

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_constants.py
"""LimX HU_D03 robot config for mjlab."""

from mjlab.actuator import BuiltinPositionActuatorCfg
from mjlab.entity import EntityArticulationInfoCfg, EntityCfg
from mjlab.utils.spec_config import CollisionCfg

from hu_d03_mjlab.robot.hu_d03_params import (
  ACTUATOR_GROUPS,
  HOME_BASE_HEIGHT,
  HOME_JOINT_POS,
)
from hu_d03_mjlab.robot.hu_d03_spec import get_spec

##
# Actuator config.
##

HU_D03_ACTUATORS = tuple(
  BuiltinPositionActuatorCfg(
    target_names_expr=g.joint_names_expr,
    stiffness=g.stiffness,
    damping=g.damping,
    effort_limit=g.effort_limit,
    armature=g.armature,
  )
  for g in ACTUATOR_GROUPS.values()
)

##
# Keyframe config.
##

HOME_KEYFRAME = EntityCfg.InitialStateCfg(
  pos=(0, 0, HOME_BASE_HEIGHT),
  joint_pos=HOME_JOINT_POS,
  joint_vel={".*": 0.0},
)

##
# Collision config.
##

FOOT_GEOMS = r"^(left|right)_foot_collision$"

# All collisions, including self collisions. Feet get condim=3, the rest condim=1.
FULL_COLLISION = CollisionCfg(
  geom_names_expr=(".*_collision.*",),
  contype=1,
  conaffinity=1,
  condim={FOOT_GEOMS: 3, ".*_collision.*": 1},
  priority={FOOT_GEOMS: 1, ".*": 0},
  friction={FOOT_GEOMS: (0.6,)},
)

##
# Final config.
##

HU_D03_ARTICULATION = EntityArticulationInfoCfg(
  actuators=HU_D03_ACTUATORS,
  soft_joint_pos_limit_factor=0.9,
)


def get_hu_d03_robot_cfg() -> EntityCfg:
  """Get a fresh HU_D03 robot configuration instance."""
  return EntityCfg(
    init_state=HOME_KEYFRAME,
    collisions=(FULL_COLLISION,),
    spec_fn=get_spec,
    articulation=HU_D03_ARTICULATION,
  )


# With the 10 Hz gains, 0.25 * effort / stiffness (the G1 rule) gives only
# 0.03-0.05 rad per unit action on the legs, because the HU_D03's reflected
# inertias are large. Use a flat 0.25 rad instead.
HU_D03_ACTION_SCALE: dict[str, float] = {".*": 0.25}

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/__init__.py
"""Registers the HU_D03 tasks with mjlab (loaded via the `mjlab.tasks` entry point)."""

from mjlab.tasks.registry import register_mjlab_task
from mjlab.tasks.tracking.rl import MotionTrackingOnPolicyRunner
from mjlab.tasks.velocity.rl import VelocityOnPolicyRunner

from .rl_cfgs import hu_d03_tracking_ppo_runner_cfg, hu_d03_velocity_ppo_runner_cfg
from .tracking_env_cfgs import hu_d03_flat_tracking_env_cfg
from .velocity_env_cfgs import hu_d03_flat_env_cfg, hu_d03_rough_env_cfg

register_mjlab_task(
  task_id="Mjlab-Velocity-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_env_cfg(),
  play_env_cfg=hu_d03_flat_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Velocity-Rough-LimX-HU-D03",
  env_cfg=hu_d03_rough_env_cfg(),
  play_env_cfg=hu_d03_rough_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_tracking_env_cfg(),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03-No-State-Estimation",
  env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False, play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/rl_cfgs.py
"""PPO runner configs for the HU_D03 tasks (G1 hyperparameters)."""

from mjlab.rl import (
  RslRlModelCfg,
  RslRlOnPolicyRunnerCfg,
  RslRlPpoAlgorithmCfg,
)


def _runner_cfg(
  experiment_name: str, entropy_coef: float, save_interval: int
) -> RslRlOnPolicyRunnerCfg:
  return RslRlOnPolicyRunnerCfg(
    actor=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
      distribution_cfg={
        "class_name": "GaussianDistribution",
        "init_std": 1.0,
        "std_type": "scalar",
      },
    ),
    critic=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
    ),
    algorithm=RslRlPpoAlgorithmCfg(
      value_loss_coef=1.0,
      use_clipped_value_loss=True,
      clip_param=0.2,
      entropy_coef=entropy_coef,
      num_learning_epochs=5,
      num_mini_batches=4,
      learning_rate=1.0e-3,
      schedule="adaptive",
      gamma=0.99,
      lam=0.95,
      desired_kl=0.01,
      max_grad_norm=1.0,
    ),
    experiment_name=experiment_name,
    save_interval=save_interval,
    num_steps_per_env=24,
    max_iterations=30_000,
  )


def hu_d03_velocity_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_velocity", entropy_coef=0.01, save_interval=50)


def hu_d03_tracking_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_tracking", entropy_coef=0.005, save_interval=100)

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/velocity_env_cfgs.py
"""LimX HU_D03 velocity environment configurations (adapted from mjlab's G1 ones)."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs import mdp as envs_mdp
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.sensor import (
  ContactMatch,
  ContactSensorCfg,
  ObjRef,
  RayCastSensorCfg,
  RingPatternCfg,
  TerrainHeightSensorCfg,
)
from mjlab.tasks.velocity import mdp
from mjlab.tasks.velocity.mdp import UniformVelocityCommandCfg
from mjlab.tasks.velocity.velocity_env_cfg import make_velocity_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"


def hu_d03_rough_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 rough terrain velocity configuration."""
  cfg = make_velocity_env_cfg()

  cfg.sim.mujoco.ccd_iterations = 500
  cfg.sim.contact_sensor_maxmatch = 500
  cfg.sim.nconmax = 70

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "terrain_scan":
      assert isinstance(sensor, RayCastSensorCfg)
      assert isinstance(sensor.frame, ObjRef)
      sensor.frame.name = PELVIS

  site_names = ("left_foot", "right_foot")

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "foot_height_scan":
      assert isinstance(sensor, TerrainHeightSensorCfg)
      sensor.frame = tuple(
        ObjRef(type="site", name=s, entity="robot") for s in site_names
      )
      sensor.pattern = RingPatternCfg.single_ring(radius=0.03, num_samples=6)

  feet_ground_cfg = ContactSensorCfg(
    name="feet_ground_contact",
    primary=ContactMatch(
      mode="subtree",
      pattern=r"^(left_ankle_roll_link|right_ankle_roll_link)$",
      entity="robot",
    ),
    secondary=ContactMatch(mode="body", pattern="terrain"),
    fields=("found", "force"),
    reduce="netforce",
    num_slots=1,
    track_air_time=True,
  )
  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (cfg.scene.sensors or ()) + (
    feet_ground_cfg,
    self_collision_cfg,
  )

  if cfg.scene.terrain is not None and cfg.scene.terrain.terrain_generator is not None:
    cfg.scene.terrain.terrain_generator.curriculum = True

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  cfg.viewer.body_name = TORSO

  twist_cmd = cfg.commands["twist"]
  assert isinstance(twist_cmd, UniformVelocityCommandCfg)
  twist_cmd.viz.z_offset = 1.75

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  # Same std rationale as the G1: loose knees/hip pitch, tight ankle roll and
  # waist roll/pitch, moderate arms. The HU_D03 also has a head and hand yaw.
  cfg.rewards["pose"].params["std_standing"] = {".*": 0.05}
  cfg.rewards["pose"].params["std_walking"] = {
    # Lower body.
    r".*hip_pitch.*": 0.3,
    r".*hip_roll.*": 0.15,
    r".*hip_yaw.*": 0.15,
    r".*knee.*": 0.35,
    r".*ankle_pitch.*": 0.25,
    r".*ankle_roll.*": 0.1,
    # Waist and head.
    r".*waist_yaw.*": 0.2,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.1,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.15,
    r".*shoulder_roll.*": 0.15,
    r".*shoulder_yaw.*": 0.1,
    r".*elbow.*": 0.15,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }
  cfg.rewards["pose"].params["std_running"] = {
    # Lower body.
    r".*hip_pitch.*": 0.5,
    r".*hip_roll.*": 0.2,
    r".*hip_yaw.*": 0.2,
    r".*knee.*": 0.6,
    r".*ankle_pitch.*": 0.35,
    r".*ankle_roll.*": 0.15,
    # Waist and head.
    r".*waist_yaw.*": 0.3,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.2,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.5,
    r".*shoulder_roll.*": 0.2,
    r".*shoulder_yaw.*": 0.15,
    r".*elbow.*": 0.35,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }

  cfg.rewards["upright"].params["asset_cfg"].body_names = (TORSO,)
  cfg.rewards["body_ang_vel"].params["asset_cfg"].body_names = (TORSO,)

  for reward_name in ["foot_clearance", "foot_slip"]:
    cfg.rewards[reward_name].params["asset_cfg"].site_names = site_names

  cfg.rewards["body_ang_vel"].weight = -0.05
  cfg.rewards["angular_momentum"].weight = -0.02
  cfg.rewards["air_time"].weight = 0.0

  cfg.rewards["self_collisions"] = RewardTermCfg(
    func=mdp.self_collision_cost,
    weight=-1.0,
    params={"sensor_name": self_collision_cfg.name, "force_threshold": 10.0},
  )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)
    cfg.terminations.pop("out_of_terrain_bounds", None)
    cfg.curriculum = {}
    cfg.events["randomize_terrain"] = EventTermCfg(
      func=envs_mdp.randomize_terrain,
      mode="reset",
      params={},
    )

    if cfg.scene.terrain is not None:
      if cfg.scene.terrain.terrain_generator is not None:
        cfg.scene.terrain.terrain_generator.curriculum = False
        cfg.scene.terrain.terrain_generator.num_cols = 5
        cfg.scene.terrain.terrain_generator.num_rows = 5
        cfg.scene.terrain.terrain_generator.border_width = 10.0

  return cfg


def hu_d03_flat_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain velocity configuration."""
  cfg = hu_d03_rough_env_cfg(play=play)

  cfg.sim.njmax = 300
  cfg.sim.mujoco.ccd_iterations = 50
  cfg.sim.contact_sensor_maxmatch = 64
  cfg.sim.nconmax = None

  assert cfg.scene.terrain is not None
  cfg.scene.terrain.terrain_type = "plane"
  cfg.scene.terrain.terrain_generator = None

  cfg.scene.sensors = tuple(
    s for s in (cfg.scene.sensors or ()) if s.name != "terrain_scan"
  )
  del cfg.observations["actor"].terms["height_scan"]
  del cfg.observations["critic"].terms["height_scan"]

  cfg.terminations.pop("out_of_terrain_bounds", None)
  cfg.curriculum.pop("terrain_levels", None)

  if play:
    twist_cmd = cfg.commands["twist"]
    assert isinstance(twist_cmd, UniformVelocityCommandCfg)
    twist_cmd.ranges.lin_vel_x = (-1.5, 2.0)
    twist_cmd.ranges.ang_vel_z = (-0.7, 0.7)

  return cfg

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/tracking_env_cfgs.py
"""LimX HU_D03 flat tracking (mimic) environment configurations."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.observation_manager import ObservationGroupCfg
from mjlab.sensor import ContactMatch, ContactSensorCfg
from mjlab.tasks.tracking.mdp import MotionCommandCfg
from mjlab.tasks.tracking.tracking_env_cfg import make_tracking_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"

# Bodies whose pose is tracked (same choice of links as the G1 config).
TRACKED_BODIES = (
  PELVIS,
  "left_hip_roll_link",
  "left_knee_link",
  "left_ankle_roll_link",
  "right_hip_roll_link",
  "right_knee_link",
  "right_ankle_roll_link",
  TORSO,
  "left_shoulder_roll_link",
  "left_elbow_link",
  "left_wrist_pitch_link",
  "right_shoulder_roll_link",
  "right_elbow_link",
  "right_wrist_pitch_link",
)


def hu_d03_flat_tracking_env_cfg(
  has_state_estimation: bool = True,
  play: bool = False,
) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain tracking configuration."""
  cfg = make_tracking_env_cfg()

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (self_collision_cfg,)

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  motion_cmd = cfg.commands["motion"]
  assert isinstance(motion_cmd, MotionCommandCfg)
  motion_cmd.anchor_body_name = TORSO
  motion_cmd.body_names = TRACKED_BODIES

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  cfg.terminations["ee_body_pos"].params["body_names"] = (
    "left_ankle_roll_link",
    "right_ankle_roll_link",
    "left_wrist_pitch_link",
    "right_wrist_pitch_link",
  )

  cfg.viewer.body_name = TORSO

  if not has_state_estimation:
    new_actor_terms = {
      k: v
      for k, v in cfg.observations["actor"].terms.items()
      if k not in ["motion_anchor_pos_b", "base_lin_vel"]
    }
    cfg.observations["actor"] = ObservationGroupCfg(
      terms=new_actor_terms,
      concatenate_terms=True,
      enable_corruption=True,
    )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)

    motion_cmd.pose_range = {}
    motion_cmd.velocity_range = {}

    motion_cmd.sampling_mode = "start"

  return cfg

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/stand_test.py
"""CPU stand test for the converted HU_D03 model (plain MuJoCo, no GPU, no mjlab).

Adds PD position actuators with the gains from hu_d03_params, puts the robot in
the home pose with its soles on the ground, and holds the pose for a few
seconds. Prints the home base height, any self-contacts at the home pose, and
how far the base drifts.

  python scripts/stand_test.py [--seconds 5] [--viewer]
"""

import argparse
import re
import sys
from pathlib import Path

import mujoco
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hu_d03_mjlab.robot.hu_d03_params import ACTUATOR_GROUPS, HOME_JOINT_POS  # noqa: E402
from hu_d03_mjlab.robot.hu_d03_spec import get_spec  # noqa: E402


def match(patterns, name):
  return any(re.fullmatch(p, name) for p in patterns)


def build_model():
  spec = get_spec()
  spec.worldbody.add_geom(
    name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=(0, 0, 1),
  )
  spec.option.timestep = 0.005
  for joint in spec.joints:
    if joint.type != mujoco.mjtJoint.mjJNT_HINGE:
      continue
    group = next(g for g in ACTUATOR_GROUPS.values() if match(g.joint_names_expr, joint.name))
    joint.armature = group.armature
    act = spec.add_actuator(name=joint.name, target=joint.name,
                            trntype=mujoco.mjtTrn.mjTRN_JOINT)
    act.set_to_position(kp=group.stiffness, kv=group.damping)
    act.forcelimited = True
    act.forcerange = (-group.effort_limit, group.effort_limit)
  return spec.compile()


def home_qpos(model):
  qpos = model.qpos0.copy()
  for j in range(1, model.njnt):
    name = model.joint(j).name
    for pattern, value in HOME_JOINT_POS.items():
      if re.fullmatch(pattern, name):
        qpos[model.jnt_qposadr[j]] = value
  # Lift or lower the base so the lowest foot site touches z = 0.
  data = mujoco.MjData(model)
  data.qpos[:] = qpos
  data.qpos[2] = 1.0
  mujoco.mj_kinematics(model, data)
  sole = min(data.site("left_foot").xpos[2], data.site("right_foot").xpos[2])
  qpos[2] = 1.0 - sole
  return qpos


def main():
  parser = argparse.ArgumentParser()
  parser.add_argument("--seconds", type=float, default=5.0)
  parser.add_argument("--viewer", action="store_true")
  args = parser.parse_args()

  model = build_model()
  data = mujoco.MjData(model)
  qpos = home_qpos(model)
  data.qpos[:] = qpos
  data.ctrl[:] = qpos[7:]
  mujoco.mj_forward(model, data)

  print(f"home base height: {qpos[2]:.4f} m, total mass {model.body_mass.sum():.2f} kg")
  floor = model.geom("floor").id
  self_contacts = [
    (model.geom(c.geom1).name, model.geom(c.geom2).name)
    for c in data.contact[: data.ncon]
    if floor not in (c.geom1, c.geom2)
  ]
  print(f"self-contacts at home pose: {self_contacts or 'none'}")

  def step():
    mujoco.mj_step(model, data)

  if args.viewer:
    from mujoco import viewer as mj_viewer

    with mj_viewer.launch_passive(model, data) as v:
      while v.is_running() and data.time < args.seconds:
        step()
        v.sync()
  else:
    while data.time < args.seconds:
      step()

  drift = np.linalg.norm(data.qpos[:2] - qpos[:2])
  up = data.xmat[model.body("base_link").id].reshape(3, 3)[:, 2]
  tilt = np.degrees(np.arccos(np.clip(up[2], -1, 1)))
  joint_err = np.abs(data.qpos[7:] - qpos[7:]).max()
  print(f"after {args.seconds:.1f}s: base z {data.qpos[2]:.4f} m, xy drift {drift:.4f} m, "
        f"tilt {tilt:.2f} deg, max joint error {joint_err:.4f} rad")
  ok = data.qpos[2] > qpos[2] - 0.1 and tilt < 10
  print("STAND TEST", "PASSED" if ok else "FAILED")
  return 0 if ok else 1


if __name__ == "__main__":
  sys.exit(main())

In [ ]:
%%writefile /kaggle/temp/hu_d03_code/hu-d03-rl/scripts/record_video.py
"""Roll out a policy headless and save an .mp4 (for Colab/Kaggle, where mjlab's
`play` viewer cannot open a window).

  python scripts/record_video.py Mjlab-Velocity-Flat-LimX-HU-D03 \
      --checkpoint-file logs/.../model_5000.pt --output-file walk.mp4
  python scripts/record_video.py Mjlab-Tracking-Flat-LimX-HU-D03 \
      --checkpoint-file ... --motion-file motion.npz --output-file mimic.mp4
  # --agent zero holds the default pose (stand test, no checkpoint needed).
  # Defaults to 1280x720; e.g. --width 1920 --height 1080 --distance 4 for 1080p.
"""

from dataclasses import asdict
from pathlib import Path
from typing import Literal

import mediapy
import torch
import tyro

import mjlab
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.tasks.tracking.mdp import MotionCommandCfg


def main(
  task_id: tyro.conf.Positional[str],
  output_file: str,
  checkpoint_file: str | None = None,
  motion_file: str | None = None,
  agent: Literal["trained", "zero"] = "trained",
  seconds: float = 10.0,
  no_terminations: bool = False,
  width: int = 1280,
  height: int = 720,
  distance: float = 3.0,
  elevation: float = -15.0,
  azimuth: float = 90.0,
):
  device = "cuda:0" if torch.cuda.is_available() else "cpu"
  env_cfg = load_env_cfg(task_id, play=True)
  agent_cfg = load_rl_cfg(task_id)
  env_cfg.scene.num_envs = 1
  # mjlab renders 320x240 from 5 m away by default; frame the whole robot instead.
  env_cfg.viewer.width, env_cfg.viewer.height = width, height
  env_cfg.viewer.distance = distance
  env_cfg.viewer.elevation = elevation
  env_cfg.viewer.azimuth = azimuth
  if no_terminations:
    env_cfg.terminations = {}

  motion_cmd = env_cfg.commands.get("motion")
  if isinstance(motion_cmd, MotionCommandCfg):
    if motion_file is None:
      raise ValueError("Tracking tasks need --motion-file.")
    motion_cmd.motion_file = motion_file

  env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")
  env = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

  if agent == "zero":
    shape = env.unwrapped.action_space.shape

    def policy(obs):
      del obs
      return torch.zeros(shape, device=env.unwrapped.device)
  else:
    if checkpoint_file is None:
      raise ValueError("--checkpoint-file is required with --agent trained.")
    runner_cls = load_runner_cls(task_id) or MjlabOnPolicyRunner
    runner = runner_cls(env, asdict(agent_cfg), device=device)
    runner.load(checkpoint_file, load_cfg={"actor": True}, strict=True,
                map_location=device)
    policy = runner.get_inference_policy(device=device)

  obs = env.get_observations()
  frames = []
  steps = int(seconds / env.unwrapped.step_dt)
  with torch.inference_mode():
    for _ in range(steps):
      obs, _, _, _ = env.step(policy(obs))
      frames.append(env.unwrapped.render())

  fps = round(1.0 / env.unwrapped.step_dt)
  Path(output_file).parent.mkdir(parents=True, exist_ok=True)
  mediapy.write_video(output_file, frames, fps=fps)
  print(f"wrote {output_file} ({len(frames)} frames at {fps} fps)")
  env.close()


if __name__ == "__main__":
  tyro.cli(main, config=mjlab.TYRO_FLAGS)

In [ ]:
sh(f"uv pip install --system -q -e {CODE / 'hu-d03-rl'}")

## 4. Sanity checks

* The stand test holds the home pose in plain MuJoCo (CPU).
* `list-envs` must show the `LimX-HU-D03` tasks. If it does not, look for a
  `[WARN] Failed to load task package 'hu_d03'` traceback.
* The zero-action video runs the full mjlab environment on the GPU: the robot should stand still.
  If the error mentions EGL, the cell switches to OSMesa and tries again. Video problems never stop the run.

In [ ]:
SCRIPTS = CODE / "hu-d03-rl" / "scripts"
sh(f"python {SCRIPTS / 'stand_test.py'} --seconds 10")
sh("list-envs | grep -i hu-d03")

In [ ]:
from IPython.display import Video, display

def video(out, *extra):
  """Record a video; returns None instead of failing the notebook."""
  cmd = f"python {SCRIPTS / 'record_video.py'} {TASK} --output-file {out} " + " ".join(extra)
  if sh(cmd, cwd=WORK, check=False) != 0:
    egl_error = "EGL" in "".join(sh.last_output[-50:])
    if not egl_error:
      print("Could not record", out, "(see the error above)")
      return None
    # OSMesa only for this command: it must not leak into training.
    print("EGL rendering failed; retrying with OSMesa.")
    subprocess.run("apt-get -qq install -y libosmesa6 > /dev/null", shell=True)
    if sh("MUJOCO_GL=osmesa " + cmd, cwd=WORK, check=False) != 0:
      print("Could not record", out, "(see the error above)")
      return None
  display(Video(str(WORK / out), embed=True, width=640))
  return WORK / out

video("videos/zero_action.mp4", "--agent zero", "--seconds 5")

## 5. Train

Logs, checkpoints (every 50 iterations) and ONNX exports go to `/kaggle/working/hu_d03_rl/logs/rsl_rl/hu_d03_velocity/<date_time>/`.
Training is stopped after `TIME_BUDGET_HOURS`; the newest checkpoint is kept.

The cell shows one progress line, updated every 10 s. The percentage is of whichever limit comes first,
`MAX_ITERATIONS` or `TIME_BUDGET_HOURS`. The full rsl_rl output goes to `hu_d03_rl/train.log`
(`!tail -n 60 /kaggle/working/hu_d03_rl/train.log` shows the latest iteration in detail).

In [ ]:
cmd = (f"MUJOCO_GL=egl train {TASK} --env.scene.num-envs {NUM_ENVS} "
       f"--agent.max-iterations {MAX_ITERATIONS} --agent.logger tensorboard")
if USE_TWO_GPUS and NUM_GPUS >= 2:
  cmd += ' --gpu-ids "[0, 1]"'
if RESUME_RUN is not None:
  cmd += f" --agent.resume True --agent.load-run {RESUME_RUN.name}"
import html, re
from datetime import timedelta
import IPython.display as ipd

class TrainProgress:
  """Parses rsl_rl's per-iteration log and shows one progress line."""
  ANSI = re.compile(r"\x1b\[[0-9;]*m")
  FIELDS = {
    "iteration": re.compile(r"Learning iteration (\d+)/(\d+)"),
    "reward": re.compile(r"Mean reward:\s+(-?[\d.]+)"),
    "length": re.compile(r"Mean episode length:\s+([\d.]+)"),
    "fps": re.compile(r"Steps per second:\s+([\d.]+)"),
  }

  def __init__(self, budget_s):
    self.budget_s = budget_s
    self.values = {}
    self.handle = ipd.display(ipd.HTML("<pre>starting training...</pre>"), display_id=True)

  def on_line(self, line):
    line = self.ANSI.sub("", line)
    for key, pattern in self.FIELDS.items():
      m = pattern.search(line)
      if m:
        self.values[key] = m.groups()

  def on_tick(self, elapsed):
    v = self.values
    if "iteration" not in v:
      text = f"setting up (compiling GPU kernels)...  {timedelta(seconds=int(elapsed))}"
    else:
      it, total = (int(x) for x in v["iteration"])
      frac = min(1.0, max((it + 1) / total, elapsed / self.budget_s))
      bar = "█" * int(frac * 30) + "░" * (30 - int(frac * 30))
      text = f"{bar} {frac * 100:5.1f}%  iter {it + 1}/{total}"
      if "reward" in v:
        text += f"  reward {float(v['reward'][0]):.2f}  ep len {float(v['length'][0]):.0f}/1000"
      if "fps" in v:
        text += f"  {float(v['fps'][0]):,.0f} steps/s"
      text += f"  {timedelta(seconds=int(elapsed))} / {timedelta(seconds=int(self.budget_s))}"
    self.handle.update(ipd.HTML(f"<pre>{html.escape(text)}</pre>"))

progress = TrainProgress(TIME_BUDGET_HOURS * 3600)
sh(cmd, cwd=WORK, max_seconds=TIME_BUDGET_HOURS * 3600, log_file=WORK / "train.log",
   on_line=progress.on_line, on_tick=progress.on_tick)

## 6. Training curves

Kaggle cannot show TensorBoard inline, so this plots the main curves from the event files.
The robot walks well when the episode length stays near the 20 s limit and `track_linear_velocity` is high.

In [ ]:
import matplotlib.pyplot as plt

def plot_curves():
  from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
  run = sorted(p for p in LOG_ROOT.glob("*/") if list(p.glob("events.*")))[-1]
  acc = EventAccumulator(str(run)); acc.Reload()
  tags = acc.Tags()["scalars"]
  wanted = [t for t in tags if t in ("Train/mean_reward", "Train/mean_episode_length")]
  wanted += [t for t in tags if "track_linear_velocity" in t or "track_angular_velocity" in t]
  fig, axes = plt.subplots(1, len(wanted), figsize=(4 * len(wanted), 3), squeeze=False)
  for ax, tag in zip(axes[0], wanted):
    ev = acc.Scalars(tag)
    ax.plot([e.step for e in ev], [e.value for e in ev])
    ax.set_title(tag.split("/")[-1], fontsize=9); ax.set_xlabel("iteration")
  fig.tight_layout()
  fig.savefig(run / "curves.png", dpi=120)
  plt.show()
  print("other tags:", [t for t in tags if t not in wanted][:40])

try:
  plot_curves()
except Exception as e:  # Never fail the run here.
  print("Could not plot curves:", e)

## 7. Video of the trained policy

In [ ]:
def latest_checkpoint():
  runs = sorted(p for p in LOG_ROOT.glob("*/") if list(p.glob("model_*.pt")))
  ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
  print("checkpoint:", ckpts[-1])
  return ckpts[-1]

try:
  video("videos/locomotion.mp4", f"--checkpoint-file {latest_checkpoint()}", "--seconds 15")
except Exception as e:
  print("Could not record the policy video:", e)

## 8. Package the results

Writes `hu_d03_velocity_<run>.zip` with the newest checkpoint, the ONNX policy, configs, curves and videos.
Download it from the notebook's *Output* tab. The full `logs/` folder is in the output too, for resuming.

In [ ]:
run = sorted(p for p in LOG_ROOT.glob("*/") if list(p.glob("model_*.pt")))[-1]
pack = WORK / "export" / run.name
shutil.rmtree(pack, ignore_errors=True)
pack.mkdir(parents=True)
newest = max(run.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
shutil.copy(newest, pack)
for f in [*run.glob("*.onnx"), *run.glob("params/*"), *run.glob("curves.png")]:
  dst = pack / f.relative_to(run)
  dst.parent.mkdir(parents=True, exist_ok=True)
  shutil.copy(f, dst)
if (WORK / "videos").exists():
  shutil.copytree(WORK / "videos", pack / "videos", dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK / f"{EXPERIMENT}_{run.name}"), "zip", pack)
print("wrote", archive)
sh(f"ls -lR {pack} | head -40")

## 9. Download everything

Zips the whole `hu_d03_rl` folder (all checkpoints, ONNX, TensorBoard logs, `train.log`, videos) into
`hu_d03_rl_all.zip` and shows a link to download it straight to your computer. The link works in an
interactive session; for a saved version, download the zip from the *Output* tab instead.

In [ ]:
from IPython.display import FileLink

everything = Path("/kaggle/working/hu_d03_rl_all")
Path(f"{everything}.zip").unlink(missing_ok=True)
shutil.make_archive(str(everything), "zip", WORK.parent, WORK.name)
size_mb = Path(f"{everything}.zip").stat().st_size / 1e6
print(f"wrote {everything}.zip ({size_mb:,.0f} MB)")
# FileLink paths are relative to the notebook's folder, /kaggle/working.
display(FileLink(os.path.relpath(f"{everything}.zip", os.getcwd())))